# Volcano Thesis — Kaggle Training Notebook (Channel Ablation)
**ResNet50 baseline vs CNN-LSTM | Thalia temporal/3 | 9-channel vs 3-channel (core geophysical)**

Combines `train.py` + `data_loader_fixed.py` + `focal_loss.py` + `statistics.json` into one notebook.

**Recipe (locked to server runs):** FocalLoss(γ=2), AdamW lr=1e-5, weight_decay=1e-4, CosineAnnealingLR (eta_min=1e-7), batch=8, grad clip 1.0, no augmentation, best-val-F1 checkpointing.

**New:** `channels: 'core'` keeps only [insar_difference, insar_coherence, dem] per timestep (3×3=9 input channels total instead of 9×3=27).

**Kaggle notes:**
- Outputs go to `/kaggle/working/outputs` — click *Save Version* to persist them.
- `/kaggle/working` is wiped between sessions. To resume a run in a new session, attach the previous version's output as a dataset and set `resume_from` in CONFIG.
- 90 epochs at 512×512 will likely NOT finish in one 12h session on a T4 — plan on resume, or reduce `image_size`/`epochs` (but then keep it identical across ALL compared runs).

## 1 — Install dependencies

In [ ]:
!pip install -q webdataset timm scikit-learn
print('deps installed')

## 2 — CONFIG (edit this cell only)

In [ ]:
CFG = {
    # --- Data ---
    'data_root'  : '/kaggle/input/datasets/bibekgautam/thalia-dataset/webdatasets/temporal/3',
    'image_size' : 512,          # 512 matches server recipe; 256 = ~4x faster but NOT comparable to server runs

    # --- Experiment ---
    'model_name'     : 'baseline',   # 'baseline' or 'cnn_lstm'
    'channels'       : 'core',       # 'all' = 9 ch/timestep, 'core' = 3 ch/timestep (phase, coherence, DEM)
    'shuffle_frames' : False,        # True = temporal-order ablation

    # --- Training (locked recipe) ---
    'epochs'        : 3,             # 3 = sanity check, 90 = full run
    'batch_size'    : 8,
    'lr'            : 1e-5,
    'weight_decay'  : 1e-4,
    'gradient_clip' : 1.0,
    'seed'          : 42,
    'num_workers'   : 2,             # drop to 0 if the loader hangs

    # --- Architecture ---
    'num_classes'       : 2,
    'timeseries_length' : 3,

    # --- Resume ---
    'resume'      : False,
    'resume_from' : '',              # path to resume_*.pth from a previous session's output dataset

    # --- Output ---
    'output_dir'  : '/kaggle/working/outputs',
}

# Derived
CORE_CHANNEL_IDX = [0, 1, 2]  # insar_difference, insar_coherence, dem (order per data_loader / Thalia utils)
CFG['n_channels_per_frame'] = 3 if CFG['channels'] == 'core' else 9

import torch
CFG['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

shuffle_tag = '_shuffled' if CFG['shuffle_frames'] else ''
CFG['run_name'] = f"{CFG['model_name']}_{CFG['channels']}{shuffle_tag}"

print(f"Device       : {CFG['device']}")
print(f"Run          : {CFG['run_name']}")
print(f"Channels     : {CFG['channels']} ({CFG['n_channels_per_frame']}/frame, "
      f"{CFG['n_channels_per_frame'] * CFG['timeseries_length']} total)")
print(f"Epochs       : {CFG['epochs']}  |  batch={CFG['batch_size']}  |  img={CFG['image_size']}")
if CFG['shuffle_frames']: print('ABLATION: frames shuffled')

## 3 — Imports, seed, output dir

In [ ]:
import io, json, random, time
from glob import glob
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import webdataset as wds
import timm
from torch.utils.data import DataLoader, IterableDataset
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import matplotlib.pyplot as plt

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
if CFG['device'].type == 'cuda':
    torch.cuda.manual_seed_all(CFG['seed'])
    print('GPU:', torch.cuda.get_device_name(0), '| count:', torch.cuda.device_count())

OUT = Path(CFG['output_dir']); OUT.mkdir(parents=True, exist_ok=True)

best_ckpt_path   = OUT / f"best_{CFG['run_name']}.pth"
resume_ckpt_path = OUT / f"resume_{CFG['run_name']}.pth"
metrics_log_path = OUT / f"metrics_{CFG['run_name']}.json"
print('outputs ->', OUT)

## 4 — Locate dataset

In [ ]:
import os
root = Path(CFG['data_root'])
if not root.exists():
    print(f"data_root not found: {root}\nSearching /kaggle/input for .tar shards...")
    for r, d, files in os.walk('/kaggle/input'):
        tars = [f for f in files if f.endswith('.tar')]
        if tars:
            print(f"{r}: {len(tars)} shards (e.g. {tars[0]})")
    raise FileNotFoundError('Fix CFG["data_root"] to the folder containing train_pos/train_neg/val/test')

max_shard = 128
for split in ['train_pos', 'train_neg', 'val', 'test']:
    n = len(glob(str(root / split / '*.tar')))
    print(f"  {split:10s}: {n:3d} shards (~{n * max_shard} samples)")

## 5 — Statistics (from statistics.json, inlined)

In [ ]:
STATS = {
    "insar_difference": {"mean": 0.001489616346722796, "std": 1.6176833645747521},
    "insar_coherence":  {"mean": 69.3472689222075,     "std": 76.50325808397666},
    "dem":              {"mean": 833.0587512223403,    "std": 977.504283216005},
    "total_column_water_vapour":        {"mean": 22.526107385139355,  "std": 12.85386775756665},
    "surface_pressure":                 {"mean": 92966.61998575741,   "std": 9132.050564654619},
    "vertical_integral_of_temperature": {"mean": 2396036.7821715856,  "std": 263503.6556952968},
}
print(f"stats loaded ({len(STATS)} entries)")

## 6 — Data loader (with channel selection)
Order of ops per sample: **normalize (all 9ch) → shuffle frames (optional) → slice core channels (optional) → resize (optional)**.
Normalizing before slicing keeps stats indexing identical to the verified server loader.

In [ ]:
# Channel order as stored in image.pth (Thalia/utilities/utils.py)
_CHANNELS_PER_TIMESTEP = [
    "insar_difference", "insar_coherence", "dem",
    "primary_date_total_column_water_vapour", "secondary_date_total_column_water_vapour",
    "primary_date_surface_pressure",          "secondary_date_surface_pressure",
    "primary_date_vertical_integral_of_temperature", "secondary_date_vertical_integral_of_temperature",
]
N_FULL = len(_CHANNELS_PER_TIMESTEP)  # 9
T = CFG['timeseries_length']

def _stats_key(name):
    for p in ("primary_date_", "secondary_date_"):
        if name.startswith(p): return name[len(p):]
    return name

def normalize(image, stats, timeseries_length=T):
    for t in range(timeseries_length):
        for c, ch in enumerate(_CHANNELS_PER_TIMESTEP):
            idx = t * N_FULL + c
            key = _stats_key(ch)
            if key in stats:
                image[idx] = (image[idx] - stats[key]['mean']) / (stats[key]['std'] + 1e-8)
    return torch.nan_to_num(image, nan=0.0, posinf=3.0, neginf=-3.0)

# Flattened indices of core channels across all timesteps, e.g. [0,1,2, 9,10,11, 18,19,20]
CORE_FLAT_IDX = [t * N_FULL + c for t in range(T) for c in CORE_CHANNEL_IDX]

def decode_sample(raw, stats, shuffle_frames=False):
    try:
        image = torch.load(io.BytesIO(raw['image.pth']),  weights_only=False).float()
        meta  = torch.load(io.BytesIO(raw['sample.pth']), weights_only=False)

        raw_label = meta.get('label', [0])
        label = int(any(raw_label) if isinstance(raw_label, (list, tuple)) else raw_label)

        image = normalize(image, stats)

        if shuffle_frames:
            perm   = torch.randperm(T)
            chunks = image.reshape(T, N_FULL, *image.shape[1:])
            image  = chunks[perm].reshape(image.shape)

        if CFG['channels'] == 'core':
            image = image[CORE_FLAT_IDX]  # (27,H,W) -> (9,H,W)

        if CFG['image_size'] != image.shape[-1]:
            image = F.interpolate(image.unsqueeze(0), size=(CFG['image_size'], CFG['image_size']),
                                  mode='bilinear', align_corners=False).squeeze(0)

        return image, torch.tensor(label, dtype=torch.long), meta
    except Exception as e:
        print(f"[decode_sample] skipping — {e}")
        return None

class _RandomMix(IterableDataset):
    def __init__(self, pos, neg): self.datasets = [pos, neg]
    def __iter__(self):
        src = [iter(d) for d in self.datasets]
        done = [False, False]
        while not all(done):
            i = random.choice([k for k, d in enumerate(done) if not d])
            try: yield next(src[i])
            except StopIteration: done[i] = True

def _collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            [b[2] for b in batch])

def create_loaders():
    dfn = lambda raw: decode_sample(raw, STATS, shuffle_frames=CFG['shuffle_frames'])
    pos = wds.WebDataset(sorted(glob(str(root/'train_pos'/'*.tar'))), shardshuffle=100).map(dfn).select(lambda x: x is not None)
    neg = wds.WebDataset(sorted(glob(str(root/'train_neg'/'*.tar'))), shardshuffle=100).map(dfn).select(lambda x: x is not None)
    train = DataLoader(_RandomMix(pos, neg), batch_size=CFG['batch_size'],
                       collate_fn=_collate, num_workers=CFG['num_workers'])
    def ev(split):
        ds = wds.WebDataset(sorted(glob(str(root/split/'*.tar'))), shardshuffle=False).map(dfn).select(lambda x: x is not None)
        return DataLoader(ds, batch_size=CFG['batch_size'], collate_fn=_collate, num_workers=CFG['num_workers'])
    return train, ev('val'), ev('test')

train_loader, val_loader, test_loader = create_loaders()
print('loaders ready')

## 7 — Sanity check one batch (verify shape + channel stats)

In [ ]:
images, labels, metas = next(iter(train_loader))
exp_ch = CFG['n_channels_per_frame'] * T
print(f"image shape : {tuple(images.shape)}  (expect [{CFG['batch_size']}, {exp_ch}, {CFG['image_size']}, {CFG['image_size']}])")
assert images.shape[1] == exp_ch, 'channel count mismatch — check channels config'
print(f"labels      : {labels.tolist()}")

names_core = ['insar_diff', 'insar_coh', 'dem']
names_all  = names_core + ['wv_p', 'wv_s', 'sp_p', 'sp_s', 'tmp_p', 'tmp_s']
names = names_core if CFG['channels'] == 'core' else names_all
print('\nPer-channel stats after normalization (sample 0, timestep 0):')
for c, name in enumerate(names):
    ch = images[0, c]
    print(f"  ch{c} ({name:10s}): mean={ch.mean():+.3f} std={ch.std():.3f} min={ch.min():+.3f} max={ch.max():+.3f}")

## 8 — Models

In [ ]:
class BaselineResNet50(nn.Module):
    """Channel-flattening baseline: all timesteps stacked as input channels."""
    def __init__(self, in_channels, num_classes=2):
        super().__init__()
        self.model = timm.create_model('resnet50', pretrained=True,
                                       num_classes=num_classes, in_chans=in_channels)
    def forward(self, x): return self.model(x)

class CNNLSTMClassifier(nn.Module):
    """CNN per frame -> LSTM over time."""
    def __init__(self, in_channels_per_frame, timeseries_len=3, lstm_hidden=256,
                 num_classes=2, dropout=0.3):
        super().__init__()
        self.T, self.C = timeseries_len, in_channels_per_frame
        self.cnn = timm.create_model('resnet50', pretrained=True, num_classes=0,
                                     global_pool='avg', in_chans=in_channels_per_frame)
        self.lstm = nn.LSTM(self.cnn.num_features, lstm_hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.LayerNorm(lstm_hidden), nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 128), nn.GELU(), nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes))
    def forward(self, x):
        B, TC, H, W = x.shape
        assert TC == self.T * self.C
        x = x.view(B * self.T, self.C, H, W)
        feats = self.cnn(x).view(B, self.T, -1)
        out, _ = self.lstm(feats)
        return self.head(out[:, -1, :])

Cf = CFG['n_channels_per_frame']
if CFG['model_name'] == 'baseline':
    model = BaselineResNet50(in_channels=Cf * T, num_classes=CFG['num_classes'])
else:
    model = CNNLSTMClassifier(in_channels_per_frame=Cf, timeseries_len=T,
                              num_classes=CFG['num_classes'])
model = model.to(CFG['device'])
print(f"{CFG['model_name']} | input ch: {Cf * T if CFG['model_name']=='baseline' else Cf}/frame "
      f"| {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

if CFG['device'].type == 'cuda' and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {torch.cuda.device_count()} GPUs")

## 9 — Focal loss, optimizer, scheduler

In [ ]:
class FocalLoss(nn.Module):
    """Copied from Thalia/losses/focal_loss.py."""
    def __init__(self, gamma=2, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma, self.reduction = gamma, reduction
        if isinstance(alpha, (float, int)): alpha = torch.Tensor([alpha, 1 - alpha])
        if isinstance(alpha, list): alpha = torch.Tensor(alpha)
        self.alpha = alpha
    def forward(self, input, target):
        target = target.unsqueeze(1)
        if input.dim() > 2:
            input = input.view(input.size(0), input.size(1), -1).transpose(1, 2)
            input = input.contiguous().view(-1, input.size(2))
        target = target.contiguous().view(-1, 1)
        logpt = F.log_softmax(input, dim=1).gather(1, target).view(-1)
        pt = logpt.data.exp()
        if self.alpha is not None:
            if self.alpha.type() != input.data.type(): self.alpha = self.alpha.type_as(input.data)
            logpt = logpt * self.alpha.gather(0, target.data.view(-1))
        loss = -1 * (1 - pt) ** self.gamma * logpt
        return loss.mean() if self.reduction == 'mean' else (loss.sum() if self.reduction == 'sum' else loss)

criterion = FocalLoss(gamma=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-7)
print('FocalLoss(γ=2) | AdamW lr=1e-5 wd=1e-4 | CosineAnnealingLR')

## 10 — Metrics, train/eval, checkpointing

In [ ]:
def compute_metrics(labels, probs):
    labels, probs = np.array(labels), np.array(probs)
    preds = (probs >= 0.5).astype(int)
    if len(np.unique(labels)) < 2:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'auroc': 50.0}
    return {'precision': precision_score(labels, preds, zero_division=0) * 100,
            'recall':    recall_score(labels, preds, zero_division=0) * 100,
            'f1':        f1_score(labels, preds, zero_division=0) * 100,
            'auroc':     roc_auc_score(labels, probs) * 100}

def train_epoch(epoch):
    model.train(); total, n = 0, 0
    for batch in train_loader:
        if batch is None: continue
        images, labels, _ = batch
        images, labels = images.to(CFG['device']), labels.to(CFG['device'])
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG['gradient_clip'])
        optimizer.step()
        total += loss.item(); n += 1
        if n % 50 == 0: print(f"  ep{epoch} batch {n} loss {loss.item():.4f}")
    return total / max(n, 1)

@torch.no_grad()
def evaluate(loader):
    model.eval(); total, n = 0, 0
    all_labels, all_probs = [], []
    for batch in loader:
        if batch is None: continue
        images, labels, _ = batch
        images, labels = images.to(CFG['device']), labels.to(CFG['device'])
        logits = model(images)
        total += criterion(logits, labels).item(); n += 1
        all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    m = compute_metrics(all_labels, all_probs)
    m['loss'] = total / max(n, 1)
    return m

def _state(m): return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()

def save_ckpt(path, epoch, best_f1, history):
    torch.save({'epoch': epoch, 'model_state_dict': _state(model),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_f1': best_f1, 'history': history, 'cfg_channels': CFG['channels']}, path)

def load_ckpt(path):
    ckpt = torch.load(path, map_location=CFG['device'], weights_only=False)
    assert ckpt.get('cfg_channels', CFG['channels']) == CFG['channels'], \
        'checkpoint was trained with a different channel config'
    tgt = model.module if isinstance(model, nn.DataParallel) else model
    tgt.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f"resumed: epoch {ckpt['epoch']} done, best_f1={ckpt['best_f1']:.2f}%")
    return ckpt['epoch'] + 1, ckpt['best_f1'], ckpt.get('history', [])

## 11 — Training loop

In [ ]:
start_epoch, best_f1, history = 1, -1.0, []

if CFG['resume']:
    rp = Path(CFG['resume_from']) if CFG['resume_from'] else resume_ckpt_path
    if rp.exists():
        start_epoch, best_f1, history = load_ckpt(rp)
    else:
        print(f'no checkpoint at {rp}, starting fresh')

t_start = time.time()
for epoch in range(start_epoch, CFG['epochs'] + 1):
    t0 = time.time()
    tr_loss = train_epoch(epoch)
    vm = evaluate(val_loader)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]
    dt = time.time() - t0

    print(f"\nEpoch {epoch}/{CFG['epochs']} | train {tr_loss:.4f} | val {vm['loss']:.4f} | "
          f"F1 {vm['f1']:.2f}% | P {vm['precision']:.2f}% | R {vm['recall']:.2f}% | "
          f"AUROC {vm['auroc']:.2f}% | lr {lr:.2e} | {dt:.0f}s")

    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': vm['loss'],
                    'val_f1': vm['f1'], 'val_precision': vm['precision'],
                    'val_recall': vm['recall'], 'val_auroc': vm['auroc'],
                    'lr': lr, 'epoch_time_sec': dt})
    with open(metrics_log_path, 'w') as f:
        json.dump({'run_name': CFG['run_name'], 'channels': CFG['channels'],
                   'image_size': CFG['image_size'], 'history': history}, f, indent=2)

    if vm['f1'] > best_f1:
        best_f1 = vm['f1']
        torch.save(_state(model), best_ckpt_path)
        print(f"  new best checkpoint (F1={best_f1:.2f}%)")
    save_ckpt(resume_ckpt_path, epoch, best_f1, history)

m, s = divmod(int(time.time() - t_start), 60)
print(f"\nDone. Best val F1: {best_f1:.2f}% | total {m}m {s}s")

## 12 — Final test evaluation (best checkpoint)

In [ ]:
best = torch.load(best_ckpt_path, map_location=CFG['device'], weights_only=True)
tgt = model.module if isinstance(model, nn.DataParallel) else model
tgt.load_state_dict(best)

tm = evaluate(test_loader)
print(f"TEST | F1 {tm['f1']:.2f}% | P {tm['precision']:.2f}% | R {tm['recall']:.2f}% | "
      f"AUROC {tm['auroc']:.2f}% | loss {tm['loss']:.4f}")

with open(metrics_log_path) as f: log = json.load(f)
log['test_metrics'] = tm
with open(metrics_log_path, 'w') as f: json.dump(log, f, indent=2)
print('saved ->', metrics_log_path)

## 13 — Training curves

In [ ]:
ep  = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [h['train_loss'] for h in history], label='train')
ax[0].plot(ep, [h['val_loss'] for h in history], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, [h['val_f1'] for h in history], label='val F1')
ax[1].plot(ep, [h['val_auroc'] for h in history], label='val AUROC')
ax[1].set_title('Validation metrics (%)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout()
plt.savefig(OUT / f"{CFG['run_name']}_curves.png", dpi=150)
plt.show()
print('files in outputs:', sorted(p.name for p in OUT.iterdir()))